In [1]:
from pathlib import Path

import mpld3
import numpy as np
from pydrake.all import (
    AddFrameTriadIllustration,
    BasicVector,
    Context,
    DiagramBuilder,
    Integrator,
    JacobianWrtVariable,
    LeafSystem,
    ModelInstanceIndex,
    MultibodyPlant,
    PiecewisePolynomial,
    PiecewisePose,
    RigidTransform,
    RotationMatrix,
    Simulator,
    StartMeshcat,
    Trajectory,
    TrajectorySource,
)

from manipulation import running_as_notebook
from manipulation.letter_generation import create_sdf_asset_from_letter
from manipulation.station import LoadScenario, MakeHardwareStation

if running_as_notebook:
    mpld3.enable_notebook()

In [2]:
# Start the visualizer.
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [3]:
def design_grasp_pose(X_WO: RigidTransform) -> tuple[RigidTransform, RigidTransform]:
    """
    fill in our code below
    """
    p_GO = [0, 0.17, 0]
    R_GO = RotationMatrix.MakeYRotation(
        - np.pi / 2.0
    ) @ RotationMatrix.MakeXRotation (- np.pi)
    X_OG = RigidTransform(R_GO, p_GO).inverse()
    X_WG = X_WO @ X_OG
    return X_OG, X_WG

In [46]:
def design_goal_poses(
    X_WO1: RigidTransform, X_WO2: RigidTransform, X_OG: RigidTransform
) -> tuple[RigidTransform, RigidTransform]:
    """
    fill in our code below. We are designing poses for two objects, so we define
    O1 as a frame centered on the first initial provided and O2 as a frame centered on the
    second initial provided.
    """
    # X_O1G1goal = RigidTransform([-0.4 - X_WO2.translation()[0], - X_WO2.translation()[1], - X_WO2.translation()[2]])
    X_O1O2 = X_WO1.inverse() @ X_WO2
    R_O2G1goal = RotationMatrix.MakeYRotation(-np.pi / 2)
    X_O2G1goal = RigidTransform(R_O2G1goal, [0, 0, -0.4])
    X_O1G1goal = X_O1O2 @ X_O2G1goal
    R_O2G2 = RotationMatrix.MakeYRotation(-np.pi / 2)
    X_O2G2goal = RigidTransform(R_O2G2)
    X_WG1goal = X_WO1 @  X_O1G1goal @  X_OG
    X_WG2goal = X_WO2 @  X_O2G2goal @  X_OG
    return X_WG1goal, X_WG2goal

def design_goal_poses2(
    X_WOref: RigidTransform, X_WO3: RigidTransform, X_WO4: RigidTransform, X_OG: RigidTransform
) -> tuple[RigidTransform, RigidTransform]:
    """
    fill in our code below. We are designing poses for two objects, so we define
    O1 as a frame centered on the first initial provided and O2 as a frame centered on the
    second initial provided.
    """
    # X_O1G1goal = RigidTransform([-0.4 - X_WO2.translation()[0], - X_WO2.translation()[1], - X_WO2.translation()[2]])
    X_O3ref = X_WO3.inverse() @ X_WOref
    R_refG1goal = RotationMatrix.MakeYRotation(-np.pi / 2)
    X_refG1goal = RigidTransform(R_refG1goal, [0, 0, +0.35])
    X_O3G1goal = X_O3ref @ X_refG1goal

    
    X_O4ref = X_WO4.inverse() @ X_WOref
    R_refG1goal = RotationMatrix.MakeYRotation(-np.pi / 2)
    X_refG1goal = RigidTransform(R_refG1goal, [0, 0, +0.7])
    X_O4G1goal = X_O4ref @ X_refG1goal


    X_WG3goal = X_WO3 @  X_O3G1goal @  X_OG
    X_WG4goal = X_WO4 @  X_O4G1goal @  X_OG
    return X_WG3goal, X_WG4goal

In [47]:
def approach_pose(X_WG: RigidTransform) -> RigidTransform:
    """
    fill in our code below
    """
    X_GGApproach = RigidTransform([0, -0.1, 0])
    X_WGApproach = X_WG @ X_GGApproach
    return X_WGApproach

In [48]:
def make_trajectory(
    X_Gs: list[RigidTransform], finger_values: np.ndarray, sample_times: list[float]
) -> tuple[Trajectory, PiecewisePolynomial]:
    robot_velocity_trajectory = None
    traj_wsg_command = None
    # TODO: define a PiecewisePose out of the X_Gs
    Traj = PiecewisePose.MakeLinear(sample_times, X_Gs)
    # TODO: set robot_velocity_trajectory to the derivative of the pose trajectory you just defined
    robot_velocity_trajectory = Traj.MakeDerivative()
    # TODO: set traj_wsg_command to a PiecewisePolynomial that commands the fingers
    traj_wsg_command = PiecewisePolynomial.FirstOrderHold(sample_times,finger_values)
    return robot_velocity_trajectory, traj_wsg_command

## Initials

In [49]:
initials = "UCBL"

In [50]:
output_dir = Path("assets/")
for letter in initials:
    create_sdf_asset_from_letter(
        text=letter,
        font_name="DejaVu Sans",
        letter_height_meters=0.2,
        extrusion_depth_meters=0.04,
        output_dir=output_dir / f"{letter}_model",
        use_bbox_collision_geometry=True,
        mass=0.1,
    )
letter_sdfs = [
    f"{Path.cwd()}/assets/{letter}_model/{letter}.sdf" for letter in initials
]

table_sdf = """<?xml version="1.0"?>
<sdf version="1.7">
  <model name="table">
    <pose>0 0 0 0 0 0</pose>
    <link name="table_link">
      <inertial>
        <mass>20</mass>
        <inertia>
          <ixx>1.0</ixx>
          <ixy>0.0</ixy>
          <ixz>0.0</ixz>
          <iyy>1.0</iyy>
          <iyz>0.0</iyz>
          <izz>1.0</izz>
        </inertia>
      </inertial>
      <collision name="box_collision">
        <geometry>
          <box>
            <size>2 2 0.1</size>
          </box>
        </geometry>
      </collision>
      <visual name="box_visual">
        <geometry>
          <box>
            <size>2 2 0.1</size>
          </box>
        </geometry>
        <material>
            <ambient>0.5 0.5 0.5 1</ambient>
            <diffuse>0.5 0.5 0.5 1</diffuse>
            <specular>0.1 0.1 0.1 1</specular>
            <emissive>0 0 0 1</emissive>
        </material>
      </visual>
    </link>
  </model>
</sdf>
"""

abs_table_sdf_path = "assets/table.sdf"
with open(abs_table_sdf_path, "w") as f:
    f.write(table_sdf)

table_sdf = f"{Path.cwd()}/assets/table.sdf"
scenario_yaml = f"""directives:
- add_model:
    name: iiwa
    file: package://drake_models/iiwa_description/sdf/iiwa7_no_collision.sdf
    default_joint_positions:
        iiwa_joint_1: [-1.57]
        iiwa_joint_2: [0.1]
        iiwa_joint_3: [0]
        iiwa_joint_4: [-1.2]
        iiwa_joint_5: [0]
        iiwa_joint_6: [ 1.6]
        iiwa_joint_7: [0]
- add_weld:
    parent: world
    child: iiwa::iiwa_link_0
- add_model:
    name: wsg
    file: package://manipulation/hydro/schunk_wsg_50_with_tip.sdf
- add_weld:
    parent: iiwa::iiwa_link_7
    child: wsg::body
    X_PC:
        translation: [0, 0, 0.09]
        rotation: !Rpy {{ deg: [90, 0, 90]}}
- add_model:
    name: table
    file: file://{table_sdf}
- add_weld:
    parent: world
    child: table::table_link
    X_PC:
        translation: [0.0, 0.0, -0.05]
        rotation: !Rpy {{ deg: [0, 0, -90] }}
- add_model:
    name: U
    file: file://{letter_sdfs[0]}
    default_free_body_pose:
        {initials[0]}_body_link:
            translation: [0.5, -0.2, 0.01]
            rotation: !Rpy {{ deg: [90, 0, 0] }}
- add_model:
    name: C
    file: file://{letter_sdfs[1]}
    default_free_body_pose:
        {initials[1]}_body_link:
            translation: [-0.2, -0.5, 0.01]
            rotation: !Rpy {{ deg: [90, 0, 90] }}
- add_model:
    name: B
    file: file://{letter_sdfs[2]}
    default_free_body_pose:
        {initials[2]}_body_link:
            translation: [-0.5, -0.0, 0.01]
            rotation: !Rpy {{ deg: [90, 0, 90] }}
- add_model:
    name: L
    file: file://{letter_sdfs[3]}
    default_free_body_pose:
        {initials[3]}_body_link:
            translation: [0.2, -0.55, 0.01]
            rotation: !Rpy {{ deg: [90, 0, 90] }}            


model_drivers:
    iiwa: !IiwaDriver
      control_mode: position_only
      hand_model_name: wsg
    wsg: !SchunkWsgDriver {{}}
"""

scenario = LoadScenario(data=scenario_yaml)

In [51]:
# Helper function to express mesh poses in terms of COM rather than geometric center


def get_initial_pose(
    plant: MultibodyPlant,
    body_name: str,
    model_instance: ModelInstanceIndex,
    plant_context: Context,
) -> RigidTransform:
    body = plant.GetBodyByName(body_name, model_instance)
    X_WS = plant.EvalBodyPoseInWorld(plant_context, body)
    X_SO = RigidTransform(body.default_spatial_inertia().get_com())
    return X_WS @ X_SO

In [52]:
def DiffIKQP(
    J_G: np.ndarray,
    V_G_desired: np.ndarray,
    q_now: np.ndarray,
    v_now: np.ndarray,
    p_now: np.ndarray,
) -> np.ndarray:
    prog = MathematicalProgram()
    v = prog.NewContinuousVariables(7, "v")
    v_max = 3.0  # do not modify

    # TODO: Add cost and constraints to prog here.
    # prog.AddL2NormCost(J_G, V_G_desired, v)
    prog.Add2NormSquaredCost(J_G, V_G_desired, v)
    prog.AddBoundingBoxConstraint(-v_max * np.ones(7), v_max * np.ones(7), v)
    solver = SnoptSolver()
    result = solver.Solve(prog)

    if not (result.is_success()):
        raise ValueError("Could not find the optimal solution.")

    v_solution = result.GetSolution(v)

    return v_solution

def DiffIKPseudoInverse(
    J_G: np.ndarray,
    V_G_desired: np.ndarray,
    q_now: np.ndarray,
    v_now: np.ndarray,
    p_now: np.ndarray,
) -> np.ndarray:
    v = np.linalg.pinv(J_G).dot(V_G_desired)
    return v

In [53]:
class PseudoInverseController(LeafSystem):
    def __init__(self, plant: MultibodyPlant):
        LeafSystem.__init__(self)
        self._plant = plant
        self._plant_context = plant.CreateDefaultContext()
        self._iiwa = plant.GetModelInstanceByName("iiwa")
        self._G = plant.GetBodyByName("body").body_frame()
        self._W = plant.world_frame()

        self.V_G_port = self.DeclareVectorInputPort("V_WG", 6)
        self.q_port = self.DeclareVectorInputPort("iiwa.position", 7)
        self.DeclareVectorOutputPort("iiwa.velocity", 7, self.CalcOutput)
        self.iiwa_start = plant.GetJointByName("iiwa_joint_1").velocity_start()
        self.iiwa_end = plant.GetJointByName("iiwa_joint_7").velocity_start()

    def CalcOutput(self, context: Context, output: BasicVector):
        """
        fill in our code below.
        """
        # evaluate the V_G_port and q_port on the current context to get those values.
        V_G_desired = self.get_input_port(0).Eval(context)
        q_now       = self.get_input_port(1).Eval(context)
        # update the positions of the internal _plant_context according to `q`.
        # HINT: you can write to a plant context by calling `self._plant.SetPositions`
        self._plant.SetPositions(self._plant_context, self._iiwa, q_now)



        # Compute the gripper jacobian
        # HINT: the jacobian is 6 x N, with N being the number of DOFs.
        # We only want the 6 x 7 submatrix corresponding to the IIWA
        J_G = self._plant.CalcJacobianSpatialVelocity(
            self._plant_context,
            JacobianWrtVariable.kQDot,
            self._G,
            [0, 0, 0],
            self._W,
            self._W,
        )
        J_G = J_G[:, 0:7]
        X_now = self._plant.CalcRelativeTransform(self._plant_context, self._W, self._G)
        p_now = X_now.translation()

        # compute `v` by mapping the gripper velocity (from the V_G_port) to the joint space
        # v = DiffIKQP(J_G, V_G_desired, q_now, v_now, p_now)
        v = np.linalg.pinv(J_G).dot(V_G_desired)
        output.SetFromVector(v)

In [54]:
# Define the builder we will use to specify the full diagram.
# Add the hardware station to the diagram
builder = DiagramBuilder()
station = MakeHardwareStation(scenario, meshcat=meshcat)
builder.AddSystem(station)
plant = station.GetSubsystemByName("plant")

# get initial poses of gripper and objects
temp_context = station.CreateDefaultContext()
temp_plant_context = plant.GetMyContextFromRoot(temp_context)
X_WGinitial = plant.EvalBodyPoseInWorld(temp_plant_context, plant.GetBodyByName("body"))
model_instance_initial1 = plant.GetModelInstanceByName("U")
model_instance_initial2 = plant.GetModelInstanceByName("C")
model_instance_initial3 = plant.GetModelInstanceByName("B")
model_instance_initial4 = plant.GetModelInstanceByName("L")
X_WO1initial = get_initial_pose(
    plant, f"{initials[0]}_body_link", model_instance_initial1, temp_plant_context
)
X_WO2initial = get_initial_pose(
    plant, f"{initials[1]}_body_link", model_instance_initial2, temp_plant_context
)
X_WO3initial = get_initial_pose(
    plant, f"{initials[2]}_body_link", model_instance_initial3, temp_plant_context
)
X_WO4initial = get_initial_pose(
    plant, f"{initials[3]}_body_link", model_instance_initial4, temp_plant_context
)

# Build trajectory keyframes
X_OG, X_WG2pick = design_grasp_pose(X_WO2initial)
_, X_WG1pick = design_grasp_pose(X_WO1initial)
_, X_WG3pick = design_grasp_pose(X_WO3initial)
_, X_WG4pick = design_grasp_pose(X_WO4initial)
X_WG1prepick, X_WG2prepick = approach_pose(X_WG1pick), approach_pose(X_WG2pick)
X_WG3prepick, X_WG4prepick = approach_pose(X_WG3pick), approach_pose(X_WG4pick)
X_WG1goal, X_WG2goal = design_goal_poses(X_WO1initial, X_WO2initial, X_OG)
X_WG3goal, X_WG4goal = design_goal_poses2(X_WO2initial, X_WO3initial, X_WO4initial, X_OG)
X_WG1pregoal, X_WG2pregoal = approach_pose(X_WG1goal), approach_pose(X_WG2goal)
X_WG3pregoal, X_WG4pregoal = approach_pose(X_WG3goal), approach_pose(X_WG4goal)


# constants for finger distances when the gripper is opened or closed
opened = 0.107
closed = 0.0

# list of keyframes, formatted as (gripper poses, finger states)
# for each object the robot starts in its default pose with its gripper open
# then it goes to the prepick pose, the pick pose, closes the gripper, and then goes
# to the place pose
keyframes = [
    (X_WGinitial, opened),
    (X_WG2prepick, opened),
    (X_WG2pick, opened),
    (X_WG2pick, closed),
    (X_WGinitial, closed),
    (X_WG2pregoal, closed),
    (X_WG2goal, closed),
    (X_WG2goal, closed),
    (X_WG2goal, opened),
    (X_WGinitial, opened),
    (X_WG1prepick, opened),
    (X_WG1pick, opened),
    (X_WG1pick, closed),
    (X_WGinitial, closed),
    (X_WG1pregoal, closed),
    (X_WG1goal, closed),
    (X_WG1goal, opened),
    (X_WGinitial, opened),
    (X_WG4prepick, opened),
    (X_WG4pick, opened),
    (X_WG4pick, closed),
    (X_WGinitial, closed),
    (X_WG4pregoal, closed),
    (X_WG4goal, closed),
    (X_WG4goal, opened),
    (X_WGinitial, opened),
    (X_WG3prepick, opened),
    (X_WG3pick, opened),
    (X_WG3pick, closed),
    (X_WGinitial, closed),
    (X_WG3pregoal, closed),
    (X_WG3goal, closed),
    (X_WG3goal, opened),
    (X_WGinitial, opened),
]

# unpack the keyframes and use them to build `Trajectory` objects
# note: we specify each keyframe as occuring 2 seconds after the last.
gripper_poses = [keyframe[0] for keyframe in keyframes]
finger_states = np.asarray([keyframe[1] for keyframe in keyframes]).reshape(1, -1)
sample_times = [2 * i for i in range(len(gripper_poses))]
traj_V_G, traj_wsg_command = make_trajectory(gripper_poses, finger_states, sample_times)

# V_G_source defines a trajectory over gripper velocities. Add it to the system.
V_G_source = builder.AddSystem(TrajectorySource(traj_V_G))
# Add the DiffIK controller we just defined to the system
controller = builder.AddSystem(PseudoInverseController(plant))
# The HardwareStation expects robot commands in terms of joint angles.
# We define the `integrator` system to map from joint_velocities to joint_angles.
integrator = builder.AddSystem(Integrator(7))
# wsg_source defines a trajectory of finger positions. Add it to the system.
wsg_source = builder.AddSystem(TrajectorySource(traj_wsg_command))

# TODO: connect the joint velocity source to the pseudoinverse controller
builder.Connect(V_G_source.get_output_port(), controller.GetInputPort("V_WG"))
# TODO: connect the controller to integrator to get joint angle commands
builder.Connect(controller.GetOutputPort("iiwa.velocity"), integrator.get_input_port())
# TODO: connect the joint angles computed by the integrateor to the iiwa.position port on the manipulation station
builder.Connect(integrator.get_output_port(), station.GetInputPort("iiwa.position"))
# TODO: connect the "iiwa.position_measured" port on the station back to the relevant input port on the controller
builder.Connect(station.GetOutputPort("iiwa.position_measured"), controller.GetInputPort("iiwa.position"))
# TODO: connect the wsg_source to the "wsg.position" input port of the station
builder.Connect(wsg_source.get_output_port(), station.GetInputPort("wsg.position"))

# visualize axes (useful for debugging)
scenegraph = station.GetSubsystemByName("scene_graph")
AddFrameTriadIllustration(
    scene_graph=scenegraph,
    body=plant.GetBodyByName(f"{initials[0]}_body_link", model_instance_initial1),
    length=0.1,
)
AddFrameTriadIllustration(
    scene_graph=scenegraph,
    body=plant.GetBodyByName(f"{initials[1]}_body_link", model_instance_initial2),
    length=0.1,
)
AddFrameTriadIllustration(
    scene_graph=scenegraph,
    body=plant.GetBodyByName(f"{initials[2]}_body_link", model_instance_initial3),
    length=0.1,
)
AddFrameTriadIllustration(
    scene_graph=scenegraph,
    body=plant.GetBodyByName(f"{initials[3]}_body_link", model_instance_initial4),
    length=0.1,
)
AddFrameTriadIllustration(
    scene_graph=scenegraph, body=plant.GetBodyByName("body"), length=0.1
)

diagram = builder.Build()

In [55]:
# Define the simulator.
simulator = Simulator(diagram)
context = simulator.get_mutable_context()
station_context = station.GetMyContextFromRoot(context)
integrator.set_integral_value(
    integrator.GetMyContextFromRoot(context),
    plant.GetPositions(
        plant.GetMyContextFromRoot(context),
        plant.GetModelInstanceByName("iiwa"),
    ),
)
diagram.ForcedPublish(context)
print(f"sanity check, simulation will run for {traj_V_G.end_time()} seconds")

# run simulation!
meshcat.StartRecording()
if running_as_notebook:
    simulator.set_target_realtime_rate(1)
simulator.AdvanceTo(traj_V_G.end_time())
meshcat.StopRecording()
meshcat.PublishRecording()

sanity check, simulation will run for 66.0 seconds


# Test UCBL